# Setup

In [1]:
from pprint import pprint


# Seed
seed = 42

# 1. Basic

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langmem import create_memory_manager
from pydantic import BaseModel


# Chat
chat = ChatOllama(
    model='qwen3:4b',
    top_k=10,
    top_p=0.9,
    temperature=1.0,
    reasoning=True,
    seed=seed,
)

# Prompt template
system_prompt = """You are a helpful assistant.
Answer precisely and shortly.
Use retrieved memory if necessary.
Ignore retrieved memory if not necessary."""

template = ChatPromptTemplate(
    messages=[
        ('system', system_prompt),
        ('human', '{retrieved_memory}'),
        ('human', '{user_prompt}'),
    ]
)

# Memory schemas
class Profile(BaseModel):
    """A user profile that contains the user name 
    and the topic user is studying."""
    
    user_name: str
    user_studying_topic: str

# Memory manager
memory_manager = create_memory_manager(
    chat,
    schemas=[Profile],
    instructions=None,
    enable_inserts=True,
    enable_updates=True,
    enable_deletes=False,
)

# Memory invoke
memories = memory_manager.invoke(
    input={
        'messages': [
            HumanMessage("My name is yana."),
            HumanMessage("I am learning memory system of NLP."),
        ],
        'max_steps':2,
    }
)

# Chat invoke
prompt = template.invoke(
    {
        'retrieved_memory': memories,
        'user_prompt': "What is my name? What do I study now?",
    }
)

response = chat.invoke(prompt)

print(f"Response: {response.text}")

Response: Your name is Yana. You are currently studying the memory system of NLP.


# 2. Memory Types

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langmem import create_memory_manager
from pydantic import BaseModel


# Chat
chat = ChatOllama(
    model='qwen3:4b',
    top_k=10,
    top_p=0.9,
    temperature=1.0,
    reasoning=True,
    seed=seed,
)

# Prompt template
system_prompt = """You are a helpful assistant.
Answer precisely and shortly.
Use retrieved memory if necessary.
Ignore retrieved memory if not necessary."""

template = ChatPromptTemplate(
    messages=[
        ('system', system_prompt),
        ('human', '{retrieved_memory}'),
        ('human', '{user_prompt}'),
    ]
)

# Memory schemas
class WorkingMemory(BaseModel):
    """A context for the current work: today's date, task."""
    date: str
    task: str

class Profile(BaseModel):
    """A user profile that contains the user name 
    and the topic user is studying."""
    
    user_name: str
    user_studying_topic: str

class Fact(BaseModel):
    """A fact: name, description."""
    name: str
    description: str

class Episode(BaseModel):
    """An episode that a user experiences: who, act, what, when."""
    who: str
    act: str
    what: str
    when: str

memory_schemas = [WorkingMemory, Profile, Fact, Episode]

# Memory manager
memory_manager = create_memory_manager(
    chat,
    schemas=memory_schemas,
    instructions=None,
    enable_inserts=True,
    enable_updates=True,
    enable_deletes=False,
)

# Memory invoke
memory_sys_prompt = f"""
You are a memory record system.
Carefully review user's messages.
Choose all the updated memory types from {memory_schemas}.
Update each memory based on user's messages.
Do not invent what does not exist in user's messages.
"""

memories = memory_manager.invoke(
    input={
        'messages': [
            SystemMessage(memory_sys_prompt),
            HumanMessage("My name is yana."),
            HumanMessage("I am learning memory system of NLP."),
            HumanMessage("I have studied NLP since last week."),
            HumanMessage("I like study cafe."),
        ],
        'max_steps':3,
    }
)

# Chat invoke
prompt = template.invoke(
    {
        'retrieved_memory': memories,
        'user_prompt': "What is my name? What do I study now?",
    }
)

response = chat.invoke(prompt)

print(f"Response: {response.text}")
print(f"Memories:\n{memories}")

Response: My name is yana.  
I study the memory system of NLP now.
Memories:
[ExtractedMemory(id='232ae136-f36a-4429-9b7a-d1666e313474', content=Profile(user_name='yana', user_studying_topic='memory system of NLP')), ExtractedMemory(id='6d44a0cd-ae38-4e07-a5eb-96527dfc7f50', content=Episode(who='yana', act='studied', what='NLP', when='last week')), ExtractedMemory(id='c28cc03f-87f0-4b3d-bd5e-44c4b020f73c', content=Fact(name='study cafe', description='I like'))]


# 3. Store

In [ ]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
import sqlite3
from langgraph.store.sqlite import SqliteStore
from uuid import uuid4
from sqlite_utils import Database
import json, re

# Setup
db_path = Path('prac/memory/memory.sqlite').resolve()
db_path.parent.mkdir(parents=True, exist_ok=True)

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
)
d_emb = embeddings._client.get_embedding_dimension()

store_conn = sqlite3.connect(
    database=db_path,
    timeout=5,
    check_same_thread=False,
    autocommit=True,
)
store = SqliteStore(
    conn=store_conn,
    index={
        'embed': embeddings,
        'dims': d_emb,
        'fields': ['$'],    # embed the entire JSON value.
    }
)
store.setup()

user_id = 'user-1'
thread_id = 'conversation-1'
namespaces = {
    WorkingMemory: (user_id, 'working', 'working_memory'),
    Profile: (user_id, 'semantic', 'profile'),
    Fact: (user_id, 'semantic', 'fact'),
    Episode: (user_id, 'episodic', 'episode'),
}
keys = {
    WorkingMemory: thread_id,
    Profile: 'profile',
}


# store.put(): add a record.
for memory in memories:
    schema = type(memory.content)
    key = keys[schema] if schema in keys else str(uuid4())
    value = memory.content.model_dump(mode='json')

    store.put(
        namespace=namespaces[schema],
        key=key,
        value=value,
    )


# store.get(): get a record.
saved = store.get(
    namespace=namespaces[Profile],
    key=keys[Profile],
)
print("Saved record:")
pprint(f"store.get(): {saved}")


# store.search(filter={...}): meta-data filtering.
filtered = store.search(
    namespaces[Profile],        # namespace.
    filter={
        'user_name': 'yana',    # {'key': 'value'} in Profile.   
    },
    limit=10,                   # retrieve up to 10 records.
)


# store.search(query=query): dense retrieval.
query = "Where do I prefer to study?"
schema = Fact
dense = store.search(
    namespaces[schema],
    query=query,
    limit=10,
)

print(f"\nDense retrieval:")
print(f"- Query: {query}")
for item in dense:
    print(f"- Item: {item.score:.4f} | {item.key} | {item.value}")


# store.list_namespaces(): Namespace list.
print(f"\nNamespace list:\n{store.list_namespaces()}")


# Delete all
store_conn.execute("DELETE FROM store")


# Close
store_conn.close()

W1021 11:03:42.401000 23052 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1021 11:03:42.477000 23052 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1021 11:03:43.528000 23052 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved record:
("store.get(): Item(namespace=['user-1', 'semantic', 'profile'], "
 "key='profile', value={'user_name': 'yana', 'user_studying_topic': 'memory "
 "system of NLP'}, created_at='2026-10-21T02:03:49', "
 "updated_at='2026-10-21T02:03:49')")

Dense retrieval:
- Query: Where do I prefer to study?
- Item: 0.2521 | 03b5a750-a202-4336-a56d-1991cc834d93 | {'name': 'study cafe', 'description': 'I like'}

Namespace list:
[('user-1', 'episodic', 'episode'), ('user-1', 'semantic', 'fact'), ('user-1', 'semantic', 'profile')]


# 4. Management

In [ ]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path
import sqlite3
from langgraph.store.sqlite import SqliteStore
from pydantic import BaseModel


# 1. Chat & Embeddings
chat = ChatOllama(
    model='qwen3:4b',
    top_k=10,
    top_p=0.9,
    temperature=1.0,
    reasoning=True,
    seed=seed,
)

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
)
d_emb = embeddings._client.get_embedding_dimension()


# 2. Store
db_path = Path('./management.sqlite').resolve()
conn = sqlite3.connect(
    database=db_path,
    check_same_thread=False,
    autocommit=True,
)
store = SqliteStore(
    conn=conn,
    index={
        'embed': embeddings,
        'dims': d_emb,
        'fields': ['$'],
    }
)
store.setup()

user_id = 'user-1'
namespace = (user_id, 'semantic', 'fact')


# 3. Memory schema
class Fact(BaseModel):
    """One explicitly stated user fact or preference."""
    text: str


# 4. Tools
